# Phase 6: Execution, Market Microstructure & Systems  
## Notebook 06_02 — Dynamic Programming Execution

### Research question

How can we formulate optimal execution as a finite-horizon dynamic programming problem where the trader adapts to inventory, time remaining, liquidity regimes, market impact, and execution risk?

This notebook follows:

```text
06_01_almgren_chriss_optimal_execution
06_02_dynamic_programming_execution
```

The previous notebook introduced the Almgren-Chriss model, where the optimal schedule can be computed analytically under simplified assumptions. This notebook generalises the execution problem using dynamic programming.

Dynamic programming is useful when execution decisions depend on state variables such as:

- inventory remaining;
- time remaining;
- liquidity regime;
- spread regime;
- volatility regime;
- volume constraints;
- urgency penalty;
- terminal liquidation penalty.

It covers:

1. execution as a control problem;
2. state, action, transition, reward/cost;
3. inventory grid;
4. liquidity regime process;
5. temporary impact;
6. spread cost;
7. inventory risk penalty;
8. terminal liquidation penalty;
9. Bellman recursion;
10. optimal liquidation policy;
11. TWAP baseline;
12. aggressive baseline;
13. passive baseline;
14. Almgren-Chriss-style baseline;
15. policy heatmaps;
16. liquidity-regime-dependent actions;
17. Monte Carlo execution simulation;
18. implementation shortfall distribution;
19. sensitivity to risk aversion;
20. sensitivity to terminal penalty;
21. participation constraints;
22. governance flags;
23. saved outputs and manifest.

The key idea:

> Dynamic programming treats execution as sequential decision-making: at every step, the trader chooses how much to trade given inventory, time, and liquidity state.

## 1. From static schedule to dynamic control

Almgren-Chriss gives a deterministic optimal schedule under stylised assumptions.

But real execution often requires state-dependent decisions:

- if liquidity is good, trade more;
- if liquidity is poor, wait if possible;
- if time is almost over, trade urgently;
- if inventory is large, reduce exposure;
- if terminal inventory remains, penalise it heavily.

Dynamic programming solves:

$$
\begin{aligned}
V_t(q, z) &= \min_a \Big[ c_t(q,a,z) \\
&\quad + E[V_{t+1}(q-a,z') \mid z] \Big]
\end{aligned}
$$

where:

- $t$ is time step;
- $q$ is inventory remaining;
- $z$ is liquidity regime;
- $a$ is shares traded now;
- $c_t$ is immediate execution and risk cost;
- $V_t$ is the value function.

## 2. State, action, transition

### State

$$
s_t=(q_t,z_t)
$$

where:

- $q_t$: inventory remaining;
- $z_t$: liquidity regime.

### Action

$$
a_t \in [0,q_t]
$$

shares sold in this interval.

### Inventory transition

$$
q_{t+1}=q_t-a_t
$$

### Liquidity transition

$$
P(z_{t+1}\mid z_t)
$$

is a Markov transition matrix.

### Terminal condition

At final time, leftover inventory is penalised heavily:

$$
V_T(q,z)=\phi q^2
$$

or forced liquidation can be implemented by infinite penalty for $q>0$.

## 3. Cost function

Immediate cost includes:

### Spread and fees

$$
c_{linear}=aS_0\frac{spread_z/2+fee}{10000}
$$

### Temporary impact

$$
c_{impact}=\eta_z a^2
$$

### Inventory risk

Remaining inventory is exposed to price moves:

$$
c_{risk}=\lambda \sigma_z^2 q_{next}^2
$$

### Participation penalty

If trading exceeds a target participation threshold, penalise:

$$
c_{participation} = \psi \max(0, a - p_{max}V_z)^2
$$

This lets the DP avoid unrealistic execution rates unless urgency demands it.

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class DPExecutionConfig:
    initial_inventory: int = 100_000
    initial_price: float = 50.0
    n_steps: int = 20
    inventory_grid_size: int = 101
    risk_aversion: float = 1.0e-7
    terminal_penalty: float = 2.0e-5
    participation_penalty: float = 1.0e-8
    max_participation: float = 0.15
    fee_bps: float = 0.20
    base_daily_vol: float = 0.025
    base_interval_volume: float = 1_000_000.0
    monte_carlo_paths: int = 10_000
    seed: int = 42
    output_subdir: str = "dynamic_programming_execution_v1"

config = DPExecutionConfig()
config

## 5. Liquidity regimes

We define three execution regimes:

| Regime | Meaning |
|---|---|
| 0 | good liquidity |
| 1 | normal liquidity |
| 2 | poor liquidity |

Each regime has:

- spread;
- temporary impact coefficient;
- volatility multiplier;
- available volume multiplier.

The regime follows a Markov chain.

In [ ]:
def liquidity_regime_parameters(config):
    params = pd.DataFrame({
        "regime": [0, 1, 2],
        "name": ["good_liquidity", "normal_liquidity", "poor_liquidity"],
        "spread_bps": [0.8, 1.8, 5.0],
        "impact_eta": [1.0e-8, 2.5e-8, 8.0e-8],
        "vol_multiplier": [0.75, 1.00, 1.80],
        "volume_multiplier": [1.60, 1.00, 0.45],
    })

    transition = np.array([
        [0.82, 0.16, 0.02],
        [0.12, 0.78, 0.10],
        [0.04, 0.24, 0.72],
    ])

    return params, transition

regime_params, transition_matrix = liquidity_regime_parameters(config)

regime_params, transition_matrix

## 6. Inventory grid and action grid

The DP operates on a discrete inventory grid.

For each inventory state $q$, allowable actions are all grid trade sizes:

$$
a \in \{0,\Delta q,2\Delta q,\ldots,q\}
$$

The grid makes the Bellman recursion simple and transparent.

In [ ]:
def make_inventory_grid(config):
    grid = np.linspace(0, config.initial_inventory, config.inventory_grid_size)
    grid = np.round(grid).astype(int)
    grid = np.unique(grid)
    return grid

inventory_grid = make_inventory_grid(config)
dq = inventory_grid[1] - inventory_grid[0]

pd.Series({
    "grid_size": len(inventory_grid),
    "dq": dq,
    "initial_inventory": config.initial_inventory,
    "first_values": inventory_grid[:5].tolist(),
    "last_values": inventory_grid[-5:].tolist(),
})

## 7. Immediate cost function

The cost function is deliberately explicit.

This makes it easy to audit which assumption drives the policy.

In [ ]:
def immediate_execution_cost(q, action, regime, config, regime_params):
    row = regime_params.loc[regime_params["regime"] == regime].iloc[0]

    spread_bps = row["spread_bps"]
    eta = row["impact_eta"]
    vol = config.base_daily_vol * row["vol_multiplier"]
    interval_volume = config.base_interval_volume * row["volume_multiplier"]

    q_next = q - action

    linear_bps = spread_bps / 2.0 + config.fee_bps
    spread_fee_cost = action * config.initial_price * linear_bps / 10000.0

    impact_cost = eta * action**2

    inventory_risk_cost = config.risk_aversion * (vol * config.initial_price)**2 * q_next**2

    max_action_by_participation = config.max_participation * interval_volume
    participation_excess = max(0.0, action - max_action_by_participation)
    participation_cost = config.participation_penalty * participation_excess**2

    total = spread_fee_cost + impact_cost + inventory_risk_cost + participation_cost

    return {
        "total_cost": total,
        "spread_fee_cost": spread_fee_cost,
        "impact_cost": impact_cost,
        "inventory_risk_cost": inventory_risk_cost,
        "participation_cost": participation_cost,
        "q_next": q_next,
        "interval_volume": interval_volume,
        "participation": action / interval_volume if interval_volume > 0 else np.nan,
    }

# Example cost.
immediate_execution_cost(50_000, 5_000, 1, config, regime_params)

## 8. Bellman recursion

For each time $t$, inventory $q$, and liquidity regime $z$:

$$
\begin{aligned}
V_t(q,z) &= \min_a \Big[ c(q,a,z) \\
&\quad + \sum_{z'}P(z'|z)V_{t+1}(q-a,z') \Big]
\end{aligned}
$$

We solve backward from terminal time.

Terminal value:

$$
V_T(q,z)=\phi q^2
$$

This strongly discourages leftover inventory.

In [ ]:
def solve_dp_execution(config, regime_params, transition_matrix):
    inventory_grid = make_inventory_grid(config)
    n_q = len(inventory_grid)
    n_z = len(regime_params)
    n_t = config.n_steps

    V = np.zeros((n_t + 1, n_q, n_z), dtype=float)
    policy_action = np.zeros((n_t, n_q, n_z), dtype=float)

    # Terminal penalty for remaining inventory.
    for iq, q in enumerate(inventory_grid):
        V[n_t, iq, :] = config.terminal_penalty * q**2

    q_to_index = {q: i for i, q in enumerate(inventory_grid)}

    for t in reversed(range(n_t)):
        for iq, q in enumerate(inventory_grid):
            allowable_actions = inventory_grid[inventory_grid <= q]

            for iz in range(n_z):
                best_value = np.inf
                best_action = 0.0

                for action in allowable_actions:
                    q_next = q - action
                    iq_next = q_to_index[int(q_next)]

                    immediate = immediate_execution_cost(q, action, iz, config, regime_params)["total_cost"]
                    continuation = transition_matrix[iz] @ V[t + 1, iq_next, :]
                    total_value = immediate + continuation

                    if total_value < best_value:
                        best_value = total_value
                        best_action = action

                V[t, iq, iz] = best_value
                policy_action[t, iq, iz] = best_action

    return V, policy_action, inventory_grid

V, policy_action, inventory_grid = solve_dp_execution(config, regime_params, transition_matrix)

V.shape, policy_action.shape

## 9. Extract optimal policy paths

We can simulate deterministic policy paths under fixed liquidity regimes.

This shows how the policy behaves when liquidity is always good, normal, or poor.

In [ ]:
def nearest_inventory_index(q, inventory_grid):
    return int(np.argmin(np.abs(inventory_grid - q)))

def deterministic_policy_path(policy_action, inventory_grid, regime, config):
    q = config.initial_inventory
    rows = []

    for t in range(config.n_steps):
        iq = nearest_inventory_index(q, inventory_grid)
        action = policy_action[t, iq, regime]
        q_next = q - action

        cost_detail = immediate_execution_cost(q, action, regime, config, regime_params)

        rows.append({
            "step": t + 1,
            "regime": regime,
            "regime_name": regime_params.loc[regime_params["regime"] == regime, "name"].iloc[0],
            "inventory_start": q,
            "action": action,
            "inventory_end": q_next,
            **{k: v for k, v in cost_detail.items() if k not in ["q_next"]},
        })
        q = q_next

    return pd.DataFrame(rows)

fixed_regime_paths = pd.concat([
    deterministic_policy_path(policy_action, inventory_grid, regime, config)
    for regime in range(len(regime_params))
], ignore_index=True)

fixed_regime_paths.head()

In [ ]:
plt.figure(figsize=(12, 6))
for regime, group in fixed_regime_paths.groupby("regime"):
    plt.plot(group["step"], group["inventory_start"] / config.initial_inventory, marker="o", label=group["regime_name"].iloc[0])
plt.title("DP Inventory Path Under Fixed Liquidity Regimes")
plt.xlabel("Step")
plt.ylabel("Fraction of inventory remaining")
plt.legend()
plt.show()

plt.figure(figsize=(12, 6))
for regime, group in fixed_regime_paths.groupby("regime"):
    plt.plot(group["step"], group["action"] / config.initial_inventory, marker="o", label=group["regime_name"].iloc[0])
plt.title("DP Trade Fraction Under Fixed Liquidity Regimes")
plt.xlabel("Step")
plt.ylabel("Fraction of initial inventory traded")
plt.legend()
plt.show()

## 10. Policy heatmap

For a selected time step, visualise:

$$
a^*(t,q,z)
$$

as a function of inventory and liquidity regime.

The policy should trade more when:

- inventory is high;
- time is short;
- liquidity is good;
- terminal penalty is near.

In [ ]:
def policy_heatmap_frame(policy_action, inventory_grid, step):
    rows = []
    for iz, row in regime_params.iterrows():
        for iq, q in enumerate(inventory_grid):
            rows.append({
                "step": step,
                "regime": int(row["regime"]),
                "regime_name": row["name"],
                "inventory": q,
                "optimal_action": policy_action[step, iq, int(row["regime"])],
                "action_fraction_of_inventory": policy_action[step, iq, int(row["regime"])] / q if q > 0 else 0.0,
            })
    return pd.DataFrame(rows)

heatmap_step_early = policy_heatmap_frame(policy_action, inventory_grid, step=2)
heatmap_step_late = policy_heatmap_frame(policy_action, inventory_grid, step=config.n_steps - 3)

heatmap_step_early.head()

In [ ]:
for step, title in [(2, "Early"), (config.n_steps - 3, "Late")]:
    frame = policy_heatmap_frame(policy_action, inventory_grid, step=step)
    pivot = frame.pivot(index="regime_name", columns="inventory", values="action_fraction_of_inventory")

    plt.figure(figsize=(12, 4))
    plt.imshow(pivot.values, aspect="auto")
    plt.colorbar(label="Action / inventory")
    plt.xticks(
        ticks=np.linspace(0, len(pivot.columns)-1, 6).astype(int),
        labels=[int(pivot.columns[i]) for i in np.linspace(0, len(pivot.columns)-1, 6).astype(int)]
    )
    plt.yticks(range(len(pivot.index)), pivot.index)
    plt.title(f"{title} Policy Heatmap: Action Fraction")
    plt.xlabel("Inventory")
    plt.ylabel("Regime")
    plt.show()

## 11. Baseline schedules

We compare DP with simple baseline policies:

1. TWAP;
2. aggressive front-loaded;
3. passive back-loaded;
4. Almgren-Chriss-style deterministic schedule.

The goal is not to declare DP always better. It is to see how adaptive control behaves under regime changes.

In [ ]:
def baseline_schedule_actions(config, name):
    n = config.n_steps
    X = config.initial_inventory

    if name == "TWAP":
        actions = np.full(n, X / n)

    elif name == "Aggressive":
        weights = np.exp(-np.linspace(0, 4, n))
        weights = weights / weights.sum()
        actions = X * weights

    elif name == "Passive":
        weights = np.linspace(0.2, 1.8, n) ** 2
        weights = weights / weights.sum()
        actions = X * weights

    elif name == "AC_Deterministic":
        temp_config = config
        kappa = np.sqrt(max(temp_config.risk_aversion * temp_config.base_daily_vol**2 / 2.5e-8, 0.0))
        t = np.linspace(0, 1, n + 1)
        if kappa < 1e-8:
            inventory = np.linspace(X, 0, n + 1)
        else:
            inventory = X * np.sinh(kappa * (1 - t)) / np.sinh(kappa)
            inventory[0] = X
            inventory[-1] = 0
        actions = inventory[:-1] - inventory[1:]

    else:
        raise ValueError("Unknown baseline")

    actions[-1] += X - actions.sum()
    return actions

baseline_names = ["TWAP", "Aggressive", "Passive", "AC_Deterministic"]

baseline_action_table = pd.DataFrame({
    name: baseline_schedule_actions(config, name)
    for name in baseline_names
})

baseline_action_table.head()

In [ ]:
plt.figure(figsize=(12, 6))
for name in baseline_names:
    inv = config.initial_inventory - np.r_[0, np.cumsum(baseline_action_table[name].values)]
    plt.plot(np.arange(config.n_steps + 1), inv / config.initial_inventory, marker="o", label=name)

normal_path = deterministic_policy_path(policy_action, inventory_grid, regime=1, config=config)
dp_inv = np.r_[config.initial_inventory, normal_path["inventory_end"].values]
plt.plot(np.arange(config.n_steps + 1), dp_inv / config.initial_inventory, marker="o", label="DP normal regime")

plt.title("Baseline Inventory Paths vs DP")
plt.xlabel("Step")
plt.ylabel("Fraction inventory remaining")
plt.legend()
plt.show()

## 12. Monte Carlo liquidity and price simulation

We now simulate execution under stochastic liquidity regimes and random price moves.

For DP:

- action depends on current inventory and regime.

For baselines:

- action schedule is fixed, but cannot sell more than remaining inventory.

Implementation shortfall is computed relative to decision price.

In [ ]:
def simulate_regime_paths(config, transition_matrix, initial_regime=1):
    rng = np.random.default_rng(config.seed + 123)
    paths = np.zeros((config.monte_carlo_paths, config.n_steps), dtype=int)
    paths[:, 0] = initial_regime

    for t in range(1, config.n_steps):
        prev = paths[:, t - 1]
        for regime in range(transition_matrix.shape[0]):
            idx = np.where(prev == regime)[0]
            if len(idx) > 0:
                paths[idx, t] = rng.choice(
                    np.arange(transition_matrix.shape[0]),
                    size=len(idx),
                    p=transition_matrix[regime],
                )

    return paths

def simulate_execution_policy(policy_name, config, regime_params, transition_matrix, policy_action=None, inventory_grid=None):
    rng = np.random.default_rng(config.seed + abs(hash(policy_name)) % 100_000)
    regime_paths = simulate_regime_paths(config, transition_matrix, initial_regime=1)

    rows = []
    path_summary = []

    baseline_actions = None
    if policy_name != "DP":
        baseline_actions = baseline_schedule_actions(config, policy_name)

    for path_id in range(config.monte_carlo_paths):
        q = config.initial_inventory
        price = config.initial_price
        proceeds = 0.0
        total_cost_components = {
            "spread_fee_cost": 0.0,
            "impact_cost": 0.0,
            "participation_cost": 0.0,
        }

        for t in range(config.n_steps):
            regime = int(regime_paths[path_id, t])
            row = regime_params.loc[regime_params["regime"] == regime].iloc[0]

            if policy_name == "DP":
                iq = nearest_inventory_index(q, inventory_grid)
                action = policy_action[t, iq, regime]
            else:
                action = baseline_actions[t]

            action = min(action, q)
            if t == config.n_steps - 1:
                action = q

            detail = immediate_execution_cost(q, action, regime, config, regime_params)

            spread_fee_per_share = detail["spread_fee_cost"] / max(action, 1.0)
            temp_impact_per_share = row["impact_eta"] * action

            exec_price = price - spread_fee_per_share - temp_impact_per_share
            proceeds += action * exec_price

            total_cost_components["spread_fee_cost"] += detail["spread_fee_cost"]
            total_cost_components["impact_cost"] += detail["impact_cost"]
            total_cost_components["participation_cost"] += detail["participation_cost"]

            q_next = q - action

            vol = config.base_daily_vol * row["vol_multiplier"]
            price_noise = vol * config.initial_price / np.sqrt(config.n_steps) * rng.standard_normal()
            permanent_impact = -2.0e-8 * action
            price = price + price_noise + permanent_impact

            rows.append({
                "path_id": path_id,
                "step": t + 1,
                "policy": policy_name,
                "regime": regime,
                "regime_name": row["name"],
                "price": price,
                "inventory_start": q,
                "action": action,
                "inventory_end": q_next,
                "exec_price": exec_price,
                "participation": detail["participation"],
                "spread_fee_cost": detail["spread_fee_cost"],
                "impact_cost": detail["impact_cost"],
                "participation_cost": detail["participation_cost"],
            })

            q = q_next

        decision_value = config.initial_inventory * config.initial_price
        shortfall = decision_value - proceeds

        path_summary.append({
            "path_id": path_id,
            "policy": policy_name,
            "proceeds": proceeds,
            "implementation_shortfall": shortfall,
            "shortfall_bps": shortfall / decision_value * 10000.0,
            "terminal_price": price,
            "terminal_inventory": q,
            **total_cost_components,
        })

    return pd.DataFrame(rows), pd.DataFrame(path_summary)

simulation_frames = []
summary_frames = []

for policy_name in ["DP"] + baseline_names:
    sim, summ = simulate_execution_policy(
        policy_name,
        config,
        regime_params,
        transition_matrix,
        policy_action=policy_action,
        inventory_grid=inventory_grid,
    )
    simulation_frames.append(sim)
    summary_frames.append(summ)

execution_simulation = pd.concat(simulation_frames, ignore_index=True)
execution_summary_paths = pd.concat(summary_frames, ignore_index=True)

execution_summary_paths.head()

## 13. Monte Carlo performance comparison

In [ ]:
execution_policy_summary = (
    execution_summary_paths
    .groupby("policy")
    .agg(
        mean_shortfall=("implementation_shortfall", "mean"),
        std_shortfall=("implementation_shortfall", "std"),
        p05_shortfall=("implementation_shortfall", lambda x: x.quantile(0.05)),
        p50_shortfall=("implementation_shortfall", lambda x: x.quantile(0.50)),
        p95_shortfall=("implementation_shortfall", lambda x: x.quantile(0.95)),
        mean_shortfall_bps=("shortfall_bps", "mean"),
        p95_shortfall_bps=("shortfall_bps", lambda x: x.quantile(0.95)),
        mean_spread_fee=("spread_fee_cost", "mean"),
        mean_impact=("impact_cost", "mean"),
        mean_participation_cost=("participation_cost", "mean"),
        mean_terminal_inventory=("terminal_inventory", "mean"),
    )
    .reset_index()
    .sort_values("mean_shortfall")
)

execution_policy_summary

In [ ]:
plt.figure(figsize=(12, 6))
for policy, group in execution_summary_paths.groupby("policy"):
    plt.hist(group["shortfall_bps"], bins=80, alpha=0.35, density=True, label=policy)
plt.title("Implementation Shortfall Distribution by Policy")
plt.xlabel("Shortfall, bps")
plt.ylabel("Density")
plt.legend()
plt.show()

## 14. Policy behaviour by liquidity regime

A useful DP policy should react to liquidity.

We inspect average actions by regime during Monte Carlo simulation.

In [ ]:
policy_regime_behavior = (
    execution_simulation
    .groupby(["policy", "regime_name"])
    .agg(
        mean_action=("action", "mean"),
        mean_action_fraction=("action", lambda x: x.mean() / config.initial_inventory),
        mean_participation=("participation", "mean"),
        p95_participation=("participation", lambda x: x.quantile(0.95)),
        n_obs=("action", "count"),
    )
    .reset_index()
    .sort_values(["policy", "regime_name"])
)

policy_regime_behavior

In [ ]:
dp_behavior = policy_regime_behavior[policy_regime_behavior["policy"] == "DP"].sort_values("mean_action")

plt.figure(figsize=(9, 5))
plt.barh(dp_behavior["regime_name"], dp_behavior["mean_action"])
plt.title("DP Mean Action by Liquidity Regime")
plt.xlabel("Mean action, shares")
plt.ylabel("Regime")
plt.show()

## 15. Sensitivity to risk aversion

Higher risk aversion should force faster execution.

We solve multiple DP policies and compare:

- mean shortfall;
- p95 shortfall;
- early liquidation speed.

In [ ]:
lambda_grid = np.logspace(-9, -5, 7)

risk_sensitivity_rows = []

for lam in lambda_grid:
    cfg = DPExecutionConfig(
        initial_inventory=config.initial_inventory,
        initial_price=config.initial_price,
        n_steps=config.n_steps,
        inventory_grid_size=config.inventory_grid_size,
        risk_aversion=float(lam),
        terminal_penalty=config.terminal_penalty,
        participation_penalty=config.participation_penalty,
        max_participation=config.max_participation,
        fee_bps=config.fee_bps,
        base_daily_vol=config.base_daily_vol,
        base_interval_volume=config.base_interval_volume,
        monte_carlo_paths=2_000,
        seed=config.seed,
    )
    V_lam, policy_lam, grid_lam = solve_dp_execution(cfg, regime_params, transition_matrix)

    normal_path = deterministic_policy_path(policy_lam, grid_lam, regime=1, config=cfg)
    half_inventory = cfg.initial_inventory / 2
    half_step = normal_path[normal_path["inventory_end"] <= half_inventory]["step"]
    half_step_value = float(half_step.iloc[0]) if len(half_step) else np.nan

    sim_lam, summ_lam = simulate_execution_policy(
        "DP",
        cfg,
        regime_params,
        transition_matrix,
        policy_action=policy_lam,
        inventory_grid=grid_lam,
    )

    risk_sensitivity_rows.append({
        "risk_aversion": lam,
        "half_liquidation_step_normal_regime": half_step_value,
        "mean_shortfall_bps": summ_lam["shortfall_bps"].mean(),
        "p95_shortfall_bps": summ_lam["shortfall_bps"].quantile(0.95),
        "std_shortfall_bps": summ_lam["shortfall_bps"].std(),
    })

risk_sensitivity = pd.DataFrame(risk_sensitivity_rows)

risk_sensitivity

In [ ]:
plt.figure(figsize=(10, 6))
plt.semilogx(risk_sensitivity["risk_aversion"], risk_sensitivity["half_liquidation_step_normal_regime"], marker="o")
plt.title("Half-Liquidation Step vs Risk Aversion")
plt.xlabel("Risk aversion")
plt.ylabel("Step when inventory falls below 50%")
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(10, 6))
plt.semilogx(risk_sensitivity["risk_aversion"], risk_sensitivity["mean_shortfall_bps"], marker="o", label="mean")
plt.semilogx(risk_sensitivity["risk_aversion"], risk_sensitivity["p95_shortfall_bps"], marker="o", label="p95")
plt.title("Shortfall vs Risk Aversion")
plt.xlabel("Risk aversion")
plt.ylabel("Shortfall, bps")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 16. Sensitivity to terminal penalty

Terminal penalty controls how strongly the DP avoids leftover inventory.

With very low terminal penalty, the policy may prefer not to complete execution if costs are high.

In real execution, incomplete execution may be unacceptable, so terminal constraints must be designed carefully.

In [ ]:
penalty_grid = np.logspace(-7, -3, 7)
terminal_sensitivity_rows = []

for penalty in penalty_grid:
    cfg = DPExecutionConfig(
        initial_inventory=config.initial_inventory,
        initial_price=config.initial_price,
        n_steps=config.n_steps,
        inventory_grid_size=config.inventory_grid_size,
        risk_aversion=config.risk_aversion,
        terminal_penalty=float(penalty),
        participation_penalty=config.participation_penalty,
        max_participation=config.max_participation,
        fee_bps=config.fee_bps,
        base_daily_vol=config.base_daily_vol,
        base_interval_volume=config.base_interval_volume,
        monte_carlo_paths=2_000,
        seed=config.seed,
    )
    V_pen, policy_pen, grid_pen = solve_dp_execution(cfg, regime_params, transition_matrix)
    normal_path = deterministic_policy_path(policy_pen, grid_pen, regime=1, config=cfg)

    terminal_sensitivity_rows.append({
        "terminal_penalty": penalty,
        "final_inventory_normal_regime": normal_path["inventory_end"].iloc[-1],
        "total_traded_normal_regime": normal_path["action"].sum(),
        "mean_action_first_5_steps": normal_path.head(5)["action"].mean(),
        "mean_action_last_5_steps": normal_path.tail(5)["action"].mean(),
    })

terminal_sensitivity = pd.DataFrame(terminal_sensitivity_rows)

terminal_sensitivity

## 17. Governance flags

A DP execution policy should be reviewed if:

1. shortfall tail is too high;
2. participation breaches the limit;
3. policy leaves residual inventory;
4. policy is too sensitive to risk aversion;
5. policy relies on uncalibrated parameters.

In [ ]:
dp_summary = execution_policy_summary[execution_policy_summary["policy"] == "DP"].iloc[0]
dp_participation = policy_regime_behavior[policy_regime_behavior["policy"] == "DP"]

shortfall_limit_bps = 75.0
participation_limit = config.max_participation
max_dp_p95_participation = dp_participation["p95_participation"].max()

risk_sensitivity_range = risk_sensitivity["mean_shortfall_bps"].max() - risk_sensitivity["mean_shortfall_bps"].min()

governance_flags = pd.DataFrame([{
    "policy": "DP",
    "mean_shortfall_bps": dp_summary["mean_shortfall_bps"],
    "p95_shortfall_bps": dp_summary["p95_shortfall_bps"],
    "max_p95_participation": max_dp_p95_participation,
    "mean_terminal_inventory": dp_summary["mean_terminal_inventory"],
    "risk_sensitivity_range_bps": risk_sensitivity_range,
    "flag_p95_shortfall_above_limit": bool(dp_summary["p95_shortfall_bps"] > shortfall_limit_bps),
    "flag_participation_above_limit": bool(max_dp_p95_participation > participation_limit),
    "flag_terminal_inventory_nonzero": bool(abs(dp_summary["mean_terminal_inventory"]) > 1e-8),
    "flag_high_risk_sensitivity": bool(risk_sensitivity_range > 30.0),
    "flag_uncalibrated_parameters": True,
}])

governance_flags["review_required"] = governance_flags[
    [
        "flag_p95_shortfall_above_limit",
        "flag_participation_above_limit",
        "flag_terminal_inventory_nonzero",
        "flag_high_risk_sensitivity",
        "flag_uncalibrated_parameters",
    ]
].any(axis=1)

governance_flags

## 18. Audit checks

Numerical and process checks:

1. value function is finite;
2. policy actions are feasible;
3. inventory never goes negative in deterministic paths;
4. baseline schedules sum to initial inventory;
5. Monte Carlo outputs are finite;
6. governance flags are present.

In [ ]:
audit_rows = []

V_finite = bool(np.isfinite(V).all())
audit_rows.append({
    "check": "value_function_finite",
    "value": V_finite,
    "passed": V_finite,
})

policy_feasible = bool((policy_action >= -1e-12).all())
for iq, q in enumerate(inventory_grid):
    policy_feasible = policy_feasible and bool((policy_action[:, iq, :] <= q + 1e-12).all())

audit_rows.append({
    "check": "policy_actions_feasible",
    "value": policy_feasible,
    "passed": policy_feasible,
})

inventory_nonnegative = bool((fixed_regime_paths["inventory_end"] >= -1e-8).all())
audit_rows.append({
    "check": "deterministic_inventory_nonnegative",
    "value": inventory_nonnegative,
    "passed": inventory_nonnegative,
})

baseline_sums_ok = True
for name in baseline_names:
    baseline_sums_ok = baseline_sums_ok and abs(baseline_schedule_actions(config, name).sum() - config.initial_inventory) < 1e-8

audit_rows.append({
    "check": "baseline_schedules_sum_to_inventory",
    "value": baseline_sums_ok,
    "passed": baseline_sums_ok,
})

mc_finite = bool(np.isfinite(execution_summary_paths.select_dtypes(include=[float, int]).to_numpy()).all())
audit_rows.append({
    "check": "monte_carlo_outputs_finite",
    "value": mc_finite,
    "passed": mc_finite,
})

governance_present = bool(len(governance_flags) == 1)
audit_rows.append({
    "check": "governance_flags_present",
    "value": governance_present,
    "passed": governance_present,
})

audit_report = pd.DataFrame(audit_rows)

audit_report

## 19. Save outputs and manifest

In [ ]:
output_dir = Path("data/processed") / config.output_subdir
output_dir.mkdir(parents=True, exist_ok=True)

regime_params.to_csv(output_dir / "regime_params.csv", index=False)
pd.DataFrame(transition_matrix).to_csv(output_dir / "transition_matrix.csv", index=False)

pd.DataFrame(V.reshape((V.shape[0], -1))).to_csv(output_dir / "value_function_flat.csv", index=False)
pd.DataFrame(policy_action.reshape((policy_action.shape[0], -1))).to_csv(output_dir / "policy_action_flat.csv", index=False)
pd.DataFrame({"inventory": inventory_grid}).to_csv(output_dir / "inventory_grid.csv", index=False)

fixed_regime_paths.to_csv(output_dir / "fixed_regime_policy_paths.csv", index=False)
heatmap_step_early.to_csv(output_dir / "policy_heatmap_early.csv", index=False)
heatmap_step_late.to_csv(output_dir / "policy_heatmap_late.csv", index=False)
baseline_action_table.to_csv(output_dir / "baseline_actions.csv", index=False)

execution_simulation.to_csv(output_dir / "execution_simulation_paths.csv", index=False)
execution_summary_paths.to_csv(output_dir / "execution_summary_paths.csv", index=False)
execution_policy_summary.to_csv(output_dir / "execution_policy_summary.csv", index=False)
policy_regime_behavior.to_csv(output_dir / "policy_regime_behavior.csv", index=False)
risk_sensitivity.to_csv(output_dir / "risk_aversion_sensitivity.csv", index=False)
terminal_sensitivity.to_csv(output_dir / "terminal_penalty_sensitivity.csv", index=False)
governance_flags.to_csv(output_dir / "governance_flags.csv", index=False)
audit_report.to_csv(output_dir / "audit_report.csv", index=False)

manifest = {
    "dataset_name": "dynamic_programming_execution_outputs",
    "schema_version": "dynamic_programming_execution_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "regime_params": regime_params.to_dict(orient="records"),
    "transition_matrix": transition_matrix.tolist(),
    "policies_compared": ["DP"] + baseline_names,
    "dp_policy_summary": dp_summary.to_dict(),
    "governance_flags": governance_flags.to_dict(orient="records"),
    "audit_passed": bool(audit_report["passed"].all()),
    "notes": [
        "This notebook formulates optimal execution as a finite-horizon dynamic programming problem.",
        "State variables are inventory and liquidity regime.",
        "Actions are discrete trade sizes on an inventory grid.",
        "The Bellman recursion minimises immediate cost plus expected continuation value.",
        "Costs include spread, fees, temporary impact, inventory risk and participation penalty.",
        "DP is compared to TWAP, aggressive, passive and AC-style deterministic baselines.",
        "Parameters are illustrative and should be calibrated before production."
    ],
}

manifest_path = output_dir / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")

output_dir / "execution_policy_summary.csv", output_dir / "policy_regime_behavior.csv", output_dir / "governance_flags.csv", manifest_path

## 20. Practical checklist for dynamic-programming execution

Before using a DP execution policy:

1. **Define state variables carefully.**
2. **Use realistic action constraints.**
3. **Calibrate impact and spread by regime.**
4. **Calibrate liquidity-regime transition probabilities.**
5. **Set terminal inventory rules explicitly.**
6. **Check participation constraints.**
7. **Compare to TWAP and VWAP baselines.**
8. **Stress risk aversion and terminal penalty.**
9. **Simulate implementation shortfall.**
10. **Add production risk controls before trading.**

## 21. Limitations

### Discrete grid approximation

The DP uses a finite inventory grid, which introduces approximation error.

### Simple regime model

Liquidity regimes are Markov states with fixed parameters. Real liquidity is richer and intraday-seasonal.

### Simplified impact

Temporary impact is quadratic in action and regime-dependent. Real impact can be nonlinear and transient.

### No limit orders

The model assumes marketable execution and does not model queue position.

### No alpha signal

The objective does not include price drift or alpha during execution.

### No adaptive volume forecast

Volume is regime-based, not forecast dynamically from real-time market data.

### Synthetic calibration

All parameters are illustrative.

## 22. Research frontier and extensions

Important extensions include:

1. continuous-action dynamic programming;
2. approximate dynamic programming;
3. reinforcement learning for execution;
4. transient impact;
5. stochastic volume curves;
6. limit order placement;
7. queue position and fill probability;
8. alpha-aware execution;
9. multi-asset basket execution;
10. futures execution with margin, tick size, night sessions and price limits.

## 23. Suggested follow-up notebooks

This notebook naturally leads to:

1. **06_03_market_impact_model_calibration**  
   Estimate impact parameters for DP and Almgren-Chriss.

2. **06_04_execution_algorithms_twap_vwap_pov**  
   Translate policies into execution algorithms.

3. **06_07_latency_and_queue_position_model**  
   Add queue and latency state variables.

4. **06_09_execution_risk_controls_and_kill_switch**  
   Wrap execution policies in safety constraints.

5. **06_11_l1_bid_ask_backtest_execution_model**  
   Use L1 bid/ask data to validate execution assumptions.

## 24. Summary

This notebook implemented dynamic programming for optimal execution.

It showed how to:

1. define execution as a finite-horizon control problem;
2. build inventory and liquidity state spaces;
3. define action constraints;
4. model spread, fees, impact, inventory risk and participation penalties;
5. solve the Bellman recursion;
6. extract optimal policies;
7. visualise policy heatmaps;
8. compare DP with TWAP, aggressive, passive and AC-style baselines;
9. simulate stochastic liquidity and price paths;
10. compare implementation shortfall distributions;
11. analyse policy behaviour by liquidity regime;
12. stress risk aversion and terminal penalties;
13. create governance flags and audit checks;
14. save outputs and manifests.

The key computational takeaway:

> Dynamic programming produces a state-dependent execution policy rather than a fixed execution schedule.

The key financial takeaway:

> The right trade size depends not only on how much inventory remains, but also on time, liquidity, risk and the cost of being wrong.

## 25. Further reading

- Almgren and Chriss, "Optimal Execution of Portfolio Transactions."
- Bertsimas and Lo, "Optimal Control of Execution Costs."
- Cartea, Jaimungal and Penalva, *Algorithmic and High-Frequency Trading.*
- Bertsekas, *Dynamic Programming and Optimal Control.*
- Powell, *Approximate Dynamic Programming.*
- Kissell, *The Science of Algorithmic Trading and Portfolio Management.*